In [144]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [145]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables) 


ConnectionError: Error connecting to server

In [ ]:
df = pd.concat([X,y], axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df = df.drop(columns=['education'])

In [ ]:
# get the duplicated rows 
df.duplicated().sum()

In [ ]:
# remove all duolicated rows
df = df.drop_duplicates()

In [ ]:
(df.isna().sum())/len(df)

In [ ]:
for col in df.columns:
    print(f"{col} = {df[col].nunique()}")

In [ ]:
df['income'].unique()

In [ ]:
df['income'] = df['income'].str.strip().str.rstrip('.')

In [ ]:
df['income'].unique()

In [ ]:
binary_map = {'<=50K' : 1, '>50K' : 0,
              'Male' : 1, 'Female' : 0}
df['sex'] = df['sex'].map(binary_map)
df['income'] = df['income'].map(binary_map)

df.head()

In [ ]:
df['race'].unique()

In [ ]:
cat_columns = ['race',
    'workclass',
    'marital-status',
    'occupation',
    'relationship',]
df = pd.get_dummies(df, columns=cat_columns, drop_first=True, dtype=int)

In [ ]:
((df['native-country']=="United-States").sum())*100/len(df)

In [ ]:
df['is_us'] = (
    df['native-country'] == 'United-States'
).astype(int)

df.drop(columns=['native-country'], inplace=True)

In [ ]:
(df.isna().sum())*100/len(df)

In [ ]:
df.describe()

In [ ]:
# visualize the distribution to check for outliers using boxplots.
X.boxplot(figsize=(15, 6))
plt.xticks(rotation=45)
plt.show()

In [ ]:
num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

# calculate the IQR
Q1 = X[num_cols].quantile(0.25)
Q3 = X[num_cols].quantile(0.75)
IQR = Q3-Q1

# calculate the lowe/upper bound

lower_bound = Q1 - (IQR*1.5)
upper_bound = Q3 + (IQR*1.5)

outliers = (X[num_cols] > upper_bound) | (X[num_cols] < lower_bound)
(outliers.sum()*100/len(X)).sort_values(ascending=False)

In [ ]:
df[num_cols].skew()

In [ ]:
# plot histo for numeic columns
df[num_cols].hist(figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
for col in skewed_cols:
    print(f'{col}')
    print('Zero values:', (df[col] == 0).sum())
    print('Unique values:', df[col].nunique())
    print('Skewness:', df[col].skew())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['income'])
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method='yeo-johnson')


X_train[['capital-gain','capital-loss']] = pt.fit_transform(
    X_train[['capital-gain','capital-loss']]
)

X_test[['capital-gain', 'capital-loss']] = pt.transform(
    X_test[['capital-gain', 'capital-loss']]
)

In [ ]:
print(X_train['capital-gain'].skew())
print(X_train['capital-loss'].skew())